In [0]:
# %sql

# -- Create a replica of target table to test above migration script
# CREATE TABLE default.dimrequesters_test
# DEEP CLONE default.dimrequesters;


In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import lit
from delta.tables import DeltaTable
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from datetime import datetime

dbutils.widgets.text("bucket", "DataStagingBucket", "S3 Bucket")
dbutils.widgets.text("env", "prod", "Environment")

# 1. Define the path to CSV file
s3_bucket = dbutils.widgets.get("bucket")
env = dbutils.widgets.get("env")

# -------------------- Edit these for csv path, target table name and schema -------------------

file_path = f"s3a://{s3_bucket}/edw_exports/{env}/dimRequesters__202603021245.csv"

target_table_name = "default.dimRequesters"

# schema mapping CSV's columns
csv_schema = StructType([
    StructField("requesterid", IntegerType(), True),
    StructField("firstname", StringType(), True),
    StructField("lastname", StringType(), True),
    StructField("middlename", StringType(), True),
    StructField("fullname", StringType(), True),
    StructField("jobtitle", StringType(), True),
    StructField("addressline1", StringType(), True),
    StructField("addressline2", StringType(), True),
    StructField("city", StringType(), True),
    StructField("zipcode", StringType(), True),
    StructField("statecode", StringType(), True),
    StructField("statename", StringType(), True),
    StructField("countryname", StringType(), True),
    StructField("workphone1", StringType(), True),
    StructField("workphone2", StringType(), True),
    StructField("mobile", StringType(), True),
    StructField("home", StringType(), True),
    StructField("fax", StringType(), True),
    StructField("email", StringType(), True),
    StructField("company", StringType(), True),
    StructField("notes", StringType(), True),
    StructField("createddate", StringType(), True),
    StructField("modifieddate", StringType(), True),
    StructField("cactive", StringType(), True),
    StructField("reasonid", StringType(), True),
    StructField("maildue", StringType(), True)
    # StructField("foirequestapplicantid", IntegerType(), True),
    # StructField("foirequestversion", IntegerType(), True),
    # StructField("axisapplicantid", IntegerType(), True),
    # StructField("requestortypeid", StringType(), True),
    # StructField("foirequestid", IntegerType(), True),
    # StructField("sourceoftruth", StringType(), True)
])

# 2. Read the CSV into a DataFrame
csv_df = (spark.read
    .format("csv")
    .option("header", "true")
    .schema(csv_schema)
    .load(file_path)
)

# 3. Add the hardcoded column to the source DataFrame
csv_df_updated = csv_df.withColumn("sourceoftruth", lit("AXIS"))

print(f"Number of rows in CSV DataFrame: {csv_df_updated.count()}")

# 3.1 Dynamically build a dictionary of the columns present in the CSV
# This creates: {"requesterid": "source.requesterid", "firstname": "source.firstname", ...}
merge_mapping = {col: f"source.{col}" for col in csv_df_updated.columns}

# 4. Upsert (Merge) into the target Delta table
if spark.catalog.tableExists(target_table_name):
    delta_table = DeltaTable.forName(spark, target_table_name)
    
    # merge condition
    merge_condition = """
        target.requesterid = source.requesterid AND 
        target.sourceoftruth = source.sourceoftruth
    """
    
    (delta_table.alias("target")
        .merge(
            csv_df_updated.alias("source"),
            merge_condition 
        )
        .whenMatchedUpdate(set = merge_mapping)
        .whenNotMatchedInsert(values = merge_mapping)
        .execute()
    )
    print("Migration complete: CSV data upserted.")

else:
    (csv_df_updated.write
        .format("delta")
        .mode("overwrite") 
        .saveAsTable(target_table_name)
    )
    print("Migration complete: New Delta table created from CSV.")